In [1]:
import sys, os
dir = os.getcwd()

ext = ['', '/..', '/../src/models', '/../src/nlp', '/../src/synth']
sys.path += [dir + i for i in ext]

### Import **API**

In [2]:
from api.football import *

### Import **Scene**

In [3]:
import string
alph = list(string.ascii_lowercase)

In [4]:
# Automatic

FOLDER = os.path.join('transcript_8/data')
demos = [i for i in os.listdir(FOLDER) if i.startswith('demonstration')]
demos.sort()

print('Found demos:', demos)

Found demos: ['demonstration0', 'demonstration1']


In [5]:
import string
from scene import Scene
from api.objects.registry import REGISTRY # TODO: Rename to ObjectsAPI

map = {}

for i, d in enumerate(demos):

    map[str(i + 1)] = {}

    folder = os.path.join(FOLDER, d, 'json_segments')
    diri = [i for i in os.listdir(folder) if i.endswith('.json')]
    diri.sort()

    for j, f in enumerate(diri):
        file = os.path.join(folder, f)
        with open(file) as f:
            data = json.load(f)

        map[str(i + 1)][alph[j]] = Scene.from_dict(data['scene'], REGISTRY)

In [6]:
narrations = '\n'.join([f"Demo {j + 1}: " + ' '.join([f"({alph[idx]}) {s.language}" for idx, s in enumerate(i.values())]) for j, i in enumerate(map.values())])
print(narrations)

Demo 1: (a) Once the goalkeeper passes the ball to [The goalkeeper passed the ball to leftBack in the scene.] the left back,you need to move down the field to create an [The leftBack passed the ball to (x: -0.5826546, y: -4.014072) in the scene.] angle to receive the ball from the left back.Once you receive the ball,pass the [The Coach passed the ball to Midfielder in the scene.] ball up the field to the midfielder.
Demo 2: (a) Once [The goalkeeper passed the ball to leftBack in the scene.] the left back receives the ball , [The leftBack passed the ball to Coach in the scene.] you need to move down the field to create an angle of pass to receive the ball from the left back [The Coach passed the ball to Midfielder in the scene.].Then pass the ball to the midfielder.


### Program Synthesis

In [7]:
from interpretable import Interpretable
from task import Task

tasks = Task.fromInterpretable(Interpretable(language=narrations))
print('\n\n'.join([str(t) + f'\nSources: {str(t.sources)}' for t in tasks]))

Task ID: 0
Objective (what): Move down the field to create an angle to receive the ball from the left back.
Control (how): None
Termination (until): None
Condition (when): Once the goalkeeper passes the ball to the left back.
Sources: ['1', '2']

Task ID: 1
Objective (what): Receive the ball from the left back.
Control (how): None
Termination (until): None
Condition (when): None
Sources: ['1', '2']

Task ID: 2
Objective (what): Pass the ball up the field to the midfielder.
Control (how): None
Termination (until): None
Condition (when): Once you receive the ball from the left back.
Sources: ['1', '2']


In [8]:
# Edit the following to modify the task specification.

# self.what = what
# self.how = how
# self.until = until
# self.when = when

# self.sources = sources

In [9]:
from action import *

act = Act()
actions = Action.fromTask(tasks, actionsAPI)
act.do(actions)

for a in act.actions:
    print(a.id)
    print(str(a.task))
    print()

Wait
Task ID: 0
Objective (what): Move down the field to create an angle to receive the ball from the left back.
Control (how): None
Termination (until): None
Condition (when): Once the goalkeeper passes the ball to the left back.

MoveTo
Task ID: 0
Objective (what): Move down the field to create an angle to receive the ball from the left back.
Control (how): None
Termination (until): None
Condition (when): Once the goalkeeper passes the ball to the left back.

Wait
Task ID: 1
Objective (what): Receive the ball from the left back.
Control (how): None
Termination (until): None
Condition (when): None

PassTo
Task ID: 2
Objective (what): Pass the ball up the field to the midfielder.
Control (how): None
Termination (until): None
Condition (when): Once you receive the ball from the left back.



In [10]:
times = [
    {
        '1': 6.0,
        '2': 0.0
    },
    {
        '1': 14.0,
        '2': 4.0
    },
    {
        '1': 18.0,
        '2': 7.0
    },
    {
        '1': 25.0,
        '2': 14.0
    }
]

In [11]:
for a, t in zip(act.actions, times):
    a.learn(map, t)

['1', '2']
[<scene.Scene object at 0x117d70290>, <scene.Scene object at 0x1260e8050>] [6.0, 0.0]
{'logic': 'A AND B', 'constraints': [{'id': 'A', 'api': 'HasBallPossession', 'params': {'ref': 'leftBack'}}, {'id': 'B', 'api': 'HasAngleOfPass', 'params': {'ref': 'Coach', 'radius': 0.0}}], 'reasoning': "The player should move down the field to create an angle to receive the ball from the left back. The condition for stopping the waiting action is when the left back has possession of the ball, which is captured by the 'HasBallPossession' constraint with the left back as the reference. Additionally, the player should create an angle to receive the ball, which is captured by the 'HasAngleOfPass' constraint with the coach as the reference. The radius is set to 0.0 to ensure a clear line of pass without interception."}
reasoning The player should move down the field to create an angle to receive the ball from the left back. The condition for stopping the waiting action is when the left back ha

In [12]:
actions_json = act.export()
print(actions_json)

{
    "actions": [
        {
            "id": "Idle",
            "args": {
                "precondition": "lambda_precondition"
            },
            "constraints": {
                "lambda_precondition": {
                    "logical": "A AND B",
                    "identifiers": [
                        "A",
                        "B"
                    ],
                    "args": {
                        "A": {
                            "type": "HasBallPossession",
                            "args": {
                                "ref": "leftBack"
                            }
                        },
                        "B": {
                            "type": "HasAngleOfPass",
                            "args": {
                                "ref": "Coach",
                                "radius": {
                                    "avg": 2.802256316999264,
                                    "std": 0.06486171001104912
                      

### **Scenic** Translation

In [13]:
def construct_coach_behavior(action_json):
    function_lines = ["behavior coachBehavior():"]
    function_lines.append("    scene = simulation()")

    for action in action_json["actions"]:
        if (action == None):
            continue
        action_id = action["id"]
        args = action.get("args", {})
        
        if action_id == "MoveTo":
            dest = args.get("dest", "")
            always = args.get("always", "")
            until = args.get("until", "")
            
            if dest and always and until:
                statement = f"    do {action_id}({dest}, {always}) until {until}"
            elif dest and until:
                statement = f"    do {action_id}({dest}) until {until}"
            elif dest:
                statement = f"    do {action_id}({dest})"
            else:
                statement = f"    do {action_id}()"
        
        elif action_id == "Idle":
            precondition = args.get("precondition", "")
            if precondition:
                statement = f"    do Idle() until {precondition}"
            else:
                statement = f"    do Idle()"

        elif action_id == "PassTo":
            obj = args.get("obj", "")
            if obj:
                statement = f"    do {action_id}({obj})"
            else:
                statement = f"    do {action_id}()"

        else:
            statement = f"    do {action_id}()"
        function_lines.append(statement)

    return "\n".join(function_lines)


In [14]:
import re

def synthesize_conditionals(expression):
    expression = re.sub(r'\bIF\s+(.*?)\s+THEN\s+(.*?)\s+ELSE\s+(.*?)\b', r'(\2 if \1 else \3)', expression)
    expression = expression.replace("AND", "and").replace("OR", "or") 
    return expression

def get_args(action, constraint_name):
    args = action.get("constraints", {}).get(constraint_name, {}).get("args", {})
    formatted_args = []
    for arg_name, details in args.items():
        arg_type = details["type"]
        arg_values = ", ".join([f"'{key}': {repr(value)}" for key, value in details["args"].items()])
        formatted_args.append(f"{arg_name} = {arg_type}({{{arg_values}}})")
    return "\n".join(formatted_args)

def create_constraint_definitions(action_index, example):
    action = example["actions"][action_index]
    definitions = []
    for constraint_name in action.get("constraints", {}):
        constraint_def = get_args(action, constraint_name)
        definitions.append(constraint_def)
    return "\n".join(definitions)

def create_lambda_dest(action):
    constraints = action.get("constraints", {}).get("lambda_dest", {})
    lambda_def = "def λ_dest(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("args", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def

def create_lambda_termination(action):
    constraints = action.get("constraints", {}).get("lambda_termination", {})
    lambda_def = "def λ_termination(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("args", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def

def create_lambda_precondition(action):
    constraints = action.get("constraints", {}).get("lambda_precondition", {})
    lambda_def = "def λ_precondition(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("args", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def


In [15]:
import json
actions_dict = json.loads(actions_json)
print(actions_dict)

{'actions': [{'id': 'Idle', 'args': {'precondition': 'lambda_precondition'}, 'constraints': {'lambda_precondition': {'logical': 'A AND B', 'identifiers': ['A', 'B'], 'args': {'A': {'type': 'HasBallPossession', 'args': {'ref': 'leftBack'}}, 'B': {'type': 'HasAngleOfPass', 'args': {'ref': 'Coach', 'radius': {'avg': 2.802256316999264, 'std': 0.06486171001104912}}}}}}}, {'id': 'MoveTo', 'args': {'dest': 'lambda_dest', 'until': ''}, 'constraints': {'lambda_dest': {'logical': 'A AND B', 'identifiers': ['A', 'B'], 'args': {'A': {'type': 'InZone', 'args': {'obj': 'Coach', 'zone': 'B2'}}, 'B': {'type': 'HasAngleOfPass', 'args': {'ref': 'leftBack', 'radius': {'avg': 1.0114652498742274, 'std': 0.004478846927278846}}}}}}}, {'id': 'Idle', 'args': {'precondition': 'lambda_precondition'}, 'constraints': {'lambda_precondition': {'logical': 'A', 'identifiers': ['A'], 'args': {'A': {'type': 'HasBallPossession', 'args': {'ref': 'leftBack'}}}}}}, {'id': 'PassTo', 'args': {'obj': 'Midfielder'}}]}


In [16]:
def generate_all_constraints_and_lambdas(example):
    for i, action in enumerate(example["actions"]):
        if (action is None):
            continue
        print(f"### Constraints and Lambda Functions for Action {i + 1}: {action['id']}")
        
        constraint_definitions = create_constraint_definitions(i, example)
        if constraint_definitions:
            print("Constraint Definitions:")
            print(constraint_definitions)
        else:
            print("No Constraint Definitions.")

        lambda_precondition_code = create_lambda_precondition(action)
        if lambda_precondition_code:
            print("\nλ_precondition Function:")
            print(lambda_precondition_code) 

        lambda_dest_code = create_lambda_dest(action)
        if lambda_dest_code:
            print("\nλ_dest Function:")
            print(lambda_dest_code)

        lambda_termination_code = create_lambda_termination(action)
        if lambda_termination_code:
            print("\nλ_termination Function:")
            print(lambda_termination_code)

        print("\n" + "=" * 40 + "\n")

generate_all_constraints_and_lambdas(actions_dict)

### Constraints and Lambda Functions for Action 1: Idle
Constraint Definitions:
A = HasBallPossession({'ref': 'leftBack'})
B = HasAngleOfPass({'ref': 'Coach', 'radius': {'avg': 2.802256316999264, 'std': 0.06486171001104912}})

λ_precondition Function:
def λ_precondition(scene, sample):
    return A(scene, sample) and B(scene, sample)


λ_dest Function:
def λ_dest(scene, sample):
    return None  # No logical expression provided


λ_termination Function:
def λ_termination(scene, sample):
    return None  # No logical expression provided



### Constraints and Lambda Functions for Action 2: MoveTo
Constraint Definitions:
A = InZone({'obj': 'Coach', 'zone': 'B2'})
B = HasAngleOfPass({'ref': 'leftBack', 'radius': {'avg': 1.0114652498742274, 'std': 0.004478846927278846}})

λ_precondition Function:
def λ_precondition(scene, sample):
    return None  # No logical expression provided


λ_dest Function:
def λ_dest(scene, sample):
    return A(scene, sample) and B(scene, sample)


λ_termination 

In [17]:
class InZone:
    def __init__(self, args):
        self.args = args

class HasAngle:
    def __init__(self, args):
        self.args = args

class IsVisible:
    def __init__(self, args):
        self.args = args

class DistanceLessThan:
    def __init__(self, args):
        self.args = args

class DistanceGreaterThan:
    def __init__(self, args):
        self.args = args


In [18]:
behavior_code = construct_coach_behavior(actions_dict)
print(behavior_code)

behavior coachBehavior():
    scene = simulation()
    do Idle() until lambda_precondition
    do MoveTo(lambda_dest)
    do Idle() until lambda_precondition
    do PassTo(Midfielder)


In [19]:
#Demo 1: (a) Once you are ready to realize that your teammate is ready to play [The expert referenced 'teammate' in the scene.] forward,then you want [The Coach passed the ball to teammate in the scene.] to pass a ball to him. (b) Once your teammate passes the ball back to you and they penetrate the [The expert referenced 'opponent_A' in the scene.] line behind [The expert referenced 'opponent_B' in the scene.] these players [The expert referenced 'opponent_C' in the scene.] [The expert referenced 'opponent_D' in the scene.],then you are ready to pass the ball back [The teammate passed the ball to Coach in the scene.] to him in order [The Coach passed the ball to teammate in the scene.] to penetrate this line of defense.

In [20]:
# Segment 1

# precondition: 
#   has angle (beginning of segment)
#   ahead of line (end of segment)
# action: pass to teammate
# termination: none

# Segment 2

# precondition: 
#   has posession (beginning of segment)
#   player ahead of defenders (end of segment)
# action: pass to teammate
# termination: none